In [ ]:
import pandas as pd
import numpy as np

import shap
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    StackingClassifier
)

from sklearn.linear_model import LogisticRegression

from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


In [ ]:
df = pd.read_csv("diabetes_binary_health_indicators_BRFSS2015 (1).csv")

df.head()


In [ ]:
# Missing values
print(df.isnull().sum())

df.fillna(df.median(), inplace=True)

# Duplicates
print(df.duplicated().sum())

df.drop_duplicates(inplace=True)

In [ ]:
X = df.drop("Diabetes_binary", axis=1)
y = df["Diabetes_binary"]

selected_features = df[
    [
        'GenHlth',
        'HighBP',
        'BMI',
        'DiffWalk',
        'HighChol',
        'Age',
        'HeartDiseaseorAttack',
        'PhysHlth',
        'MentHlth',
        'Income'
    ]
]

selected_features.head()


In [ ]:
X = selected_features

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
base_models = [
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('gb', GradientBoostingClassifier(random_state=42))
]

In [ ]:

stack_model = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(),
    cv=5
)

stack_model.fit(X_train_scaled, y_train)

In [ ]:
y_pred = stack_model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)

print(f"\nAccuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

plt.title("Confusion Matrix - Stacking Classifier")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
!pip install lime -q

from lime.lime_tabular import LimeTabularExplainer

lime_explainer = LimeTabularExplainer(
    training_data=X_train_scaled,
    feature_names=X.columns.tolist(),
    class_names=['No Diabetes', 'Diabetes'],
    mode='classification'
)

lime_exp = lime_explainer.explain_instance(
    X_test_scaled[0],
    stack_model.predict_proba,
    num_features=10
)

lime_exp.show_in_notebook(show_table=True)


In [ ]:
feature_names = X.columns.tolist()

features_to_plot = ['GenHlth', 'HighBP', 'BMI']

fig, axes = plt.subplots(1, 3, figsize=(18,5))

for idx, feature in enumerate(features_to_plot):

    values = np.linspace(
        X_train[feature].min(),
        X_train[feature].max(),
        50
    )

    probs = []

    for val in values:

        X_temp = X_train.copy()

        X_temp[feature] = val

        X_temp_scaled = scaler.transform(X_temp)

        prob = stack_model.predict_proba(X_temp_scaled)[:,1].mean()

        probs.append(prob)

    axes[idx].plot(values, probs)

    axes[idx].set_title(f"PDP - {feature}")

    axes[idx].set_xlabel(feature)

    axes[idx].set_ylabel("Average Prediction")

plt.tight_layout()

plt.show()


In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    stack_model,
    X_test_scaled,
    y_test,
    n_repeats=10,
    random_state=42
)

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": perm.importances_mean
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("\nPermutation Importance:")
print(importance_df)

plt.figure(figsize=(10,6))

sns.barplot(
    data=importance_df,
    x="Importance",
    y="Feature"
)

plt.title("Permutation Importance - Stacking Classifier")

plt.show()

In [ ]:
from alibi.explainers import ALE

# ALE explainer
ale_knn = ALE(
    knn_model.predict_proba,
    feature_names=X.columns.tolist()
)

# explanation
exp_knn = ale_knn.explain(X.values)

# plot
ale_knn.plot(exp_knn, features=[0, 1, 2])

plt.title("ALE Plot - KNN")
plt.show()